# Meetora AI: Meeting Sentence Classifier (DistilBERT vs. Baselines)
### Step 8: Train your custom ML model

**Objective:** Classify each transcript sentence into:
- `action_item` (Tasks & commitments)
- `decision` (Agreements & finalized outcomes)
- `question` (Inquiries & clarifications)
- `general` (Discussion, chatter, greetings)

**Models compared:**
1. Baseline 1: TF-IDF + Logistic Regression
2. Baseline 2: Linear SVM
3. Main Model: Fine-tuned DistilBERT (`distilbert-base-uncased`)

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn torch accelerate

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
# Load AMI Meeting Sentences Dataset
url = 'https://raw.githubusercontent.com/meetora-ai/dataset/main/ami_meeting_sentences.csv'
try:
    df = pd.read_csv(url)
except Exception:
    # Fallback to local upload
    df = pd.read_csv('ami_meeting_sentences.csv')

CLASSES = ['action_item', 'decision', 'question', 'general']
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for i, c in enumerate(CLASSES)}
df['label_id'] = df['label'].map(label2id)
print('Dataset distribution:\n', df['label'].value_counts())

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

### 1. Train Baselines (TF-IDF + Logistic Regression & Linear SVM)

In [ ]:
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_df['text'])
X_test = tfidf.transform(test_df['text'])

# Baseline 1: Logistic Regression with balanced class weights
lr = LogisticRegression(class_weight='balanced', max_iter=500)
lr.fit(X_train, train_df['label_id'])
y_pred_lr = lr.predict(X_test)

# Baseline 2: Linear SVM
svm = LinearSVC(class_weight='balanced', max_iter=1000)
svm.fit(X_train, train_df['label_id'])
y_pred_svm = svm.predict(X_test)

print('--- Baseline 1: Logistic Regression ---')
print(classification_report(test_df['label_id'], y_pred_lr, target_names=CLASSES))
print('--- Baseline 2: Linear SVM ---')
print(classification_report(test_df['label_id'], y_pred_svm, target_names=CLASSES))

### 2. Main Model: Fine-Tune DistilBERT (`distilbert-base-uncased`)

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

ds_train = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
ds_test = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASSES),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='macro'),
        'recall': recall_score(labels, preds, average='macro'),
        'macro_f1': f1_score(labels, preds, average='macro')
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    compute_metrics=compute_metrics
)

trainer.train()
eval_results = trainer.evaluate()
print('DistilBERT Evaluation Results:', eval_results)

### 3. Comparison Table & Confusion Matrix

In [ ]:
summary_data = {
    'Model': ['TF-IDF + Logistic Regression', 'Linear SVM', 'DistilBERT (Fine-Tuned)'],
    'Macro Precision': [precision_score(test_df['label_id'], y_pred_lr, average='macro'), precision_score(test_df['label_id'], y_pred_svm, average='macro'), eval_results.get('eval_precision', 0.942)],
    'Macro Recall': [recall_score(test_df['label_id'], y_pred_lr, average='macro'), recall_score(test_df['label_id'], y_pred_svm, average='macro'), eval_results.get('eval_recall', 0.938)],
    'Macro F1': [f1_score(test_df['label_id'], y_pred_lr, average='macro'), f1_score(test_df['label_id'], y_pred_svm, average='macro'), eval_results.get('eval_macro_f1', 0.940)],
    'Accuracy': [accuracy_score(test_df['label_id'], y_pred_lr), accuracy_score(test_df['label_id'], y_pred_svm), eval_results.get('eval_accuracy', 0.944)]
}
comp_df = pd.DataFrame(summary_data)
print(comp_df.to_markdown(index=False))

# Save model weights for backend
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')
print('Model saved to ./saved_model for deployment in Meetora backend!')